# Working Memory & Context Window Management

> Actively prioritizing, pinning, and evicting context elements so the most relevant information occupies the limited context window.

Imagine you're a waiter in a busy restaurant. You can hold maybe four orders in your head at once. When a fifth table flags you down, you have to write something down or forget an earlier order. The skill isn't memorizing everything. It's deciding *what to keep in your head* and *what to offload*.

Every LLM operates within a finite context window (the text it can "see" at once). Whether it's 4,000 or a million tokens, there's always a boundary. What lives inside that boundary determines reasoning quality.

A naive approach treats the context window as a queue. Oldest messages fall off the end. This is a poor strategy. A critical instruction from the beginning of a conversation can vanish while irrelevant chatter remains. This is the "lost in the middle" problem (Liu et al., 2023). Models struggle to attend to information buried in the center of long contexts.

Working Memory & Context Window Management draws from cognitive psychology. Baddeley's model of human working memory describes a limited-capacity system with an attentional controller. That controller decides what stays in active awareness and what fades. Agents need the same thing: a management layer that scores, prioritizes, pins (protects), and evicts (removes) context elements.

**What you'll build in this notebook:**
1. A `ContextItem` data model with token counts and salience scores.
2. A `SalienceScorer` that ranks items by relevance, recency, and importance.
3. An `EvictionEngine` with two policies: LRU and importance-weighted LRU.
4. A `ContextWindowManager` that orchestrates the full lifecycle.
5. An end-to-end demo showing context management across a long conversation.

## Key Concepts

- **Context window**: The text an LLM can "see" at once. Every model has a maximum token limit. Everything the model reads and reasons over must fit inside this window.
- **Token**: A word-piece the model processes. "Chatbot" might be one token; "unbelievable" might be three. Token counts determine cost and capacity.
- **Attention management**: Controlling which information occupies the context window at any moment. This replaces passive accumulation with intentional curation.
- **Context prioritization**: Assigning scores to each context element based on relevance to the current task, recency of use, and inherent importance.
- **Pinned context**: Items protected from eviction no matter what. System prompts, core instructions, and user preferences live here. Think of these as permanent sticky notes on your desk.
- **Dynamic eviction policies**: Strategies for removing low-value context when capacity is reached. LRU (Least Recently Used) removes the item untouched longest. Importance-based eviction removes the lowest-scored item instead.
- **Salience scoring**: Computing a number that reflects how important a context item is for the agent's *current* reasoning step. We'll combine recency decay and importance weight.
- **Embedding**: A list of numbers (a vector) that captures the meaning of a text. Texts with similar meaning have similar embeddings. We use embeddings to measure relevance.
- **Cosine similarity**: A way to measure how close two embeddings are. Values range from -1 (opposite) to 1 (identical meaning).
- **External memory store**: A place to archive evicted items so they're retrievable later. Items leave active context but aren't lost forever.

## Architecture

<p align="center">
  <img src="../../images/diagrams/12_working_memory_context_window.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    IS["Info Sources\n(User, Tools, Memory)"] --> SS["Salience Scorer"]
    SS --> PQ["Priority Queue"]
    PQ --> CW["Context Window"]

    subgraph CW["Context Window"]
        PZ["Pinned Zone\n(System prompt,\ncore instructions)"]
        DZ["Dynamic Zone\n(Scored & ranked\ncontext items)"]
    end

    CW --> LLM["LLM"]
    DZ --> EE["Eviction Engine"]
    EE --> EMS["External Memory Store"]
    EMS -->|"Retrieval on demand"| SS

    style PZ fill:#2d5a2d,stroke:#4a9,color:#fff
    style DZ fill:#2d4a7a,stroke:#49a,color:#fff
    style EE fill:#7a2d2d,stroke:#a44,color:#fff
```

</details>

**Data flow**: Information from various sources (user messages, tool outputs, retrieved memories) enters the **Salience Scorer**. The scorer evaluates relevance and importance. Scored items join the **Priority Queue**.

The **Context Window** has two zones. The **Pinned Zone** holds protected, always-present items like the system prompt. The **Dynamic Zone** holds items ranked by salience score.

When the window approaches capacity, the **Eviction Engine** removes the lowest-scoring items from the Dynamic Zone. It archives them in the **External Memory Store**. When previously evicted information becomes relevant again, the retrieval loop brings it back through the Salience Scorer.

## Setup

Install dependencies. We use:
- `openai` for chat completions and embeddings
- `tiktoken` for accurate token counting (a library that splits text into the exact tokens the model uses)
- `numpy` for vector math
- `python-dotenv` to load API keys from a `.env` file

In [ ]:
%pip install -q openai tiktoken numpy python-dotenv

Import all libraries and create the API client. You need an `OPENAI_API_KEY` in your `.env` file.

In [ ]:
import os
import time
import json
import hashlib
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()  # reads OPENAI_API_KEY from env
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

# We'll use this encoder for token counting throughout the notebook
ENCODER = tiktoken.encoding_for_model("gpt-4o-mini")

CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

print("Setup complete.")

## Implementation

### Step 1: The `ContextItem` Data Model

Think of a library card catalog. Each card describes a book and carries metadata: title, last checkout date, how many times it's been borrowed. Our `ContextItem` works the same way. It wraps a piece of text with the metadata the manager needs to make eviction decisions.

Each item tracks its content, role, token count, salience score, timestamps, and whether it's pinned.

In [ ]:
def count_tokens(text: str) -> int:
    """Count tokens using tiktoken."""
    return len(ENCODER.encode(text))


@dataclass
class ContextItem:
    """A single item in the context window with management metadata."""
    content: str
    role: str  # "system", "user", "assistant", "tool"
    token_count: int = 0
    salience_score: float = 0.0
    created_at: float = 0.0
    last_accessed: float = 0.0
    access_count: int = 0
    pinned: bool = False
    source: str = "user"  # user, tool, memory, system
    item_id: str = ""
    embedding: Optional[list[float]] = field(default=None, repr=False)

    def __post_init__(self):
        if not self.token_count:
            self.token_count = count_tokens(self.content)
        if not self.created_at:
            self.created_at = time.time()
        if not self.last_accessed:
            self.last_accessed = self.created_at
        if not self.item_id:
            # Deterministic ID from content + role
            raw = f"{self.role}:{self.content}"
            self.item_id = hashlib.sha256(raw.encode()).hexdigest()[:12]

    def touch(self):
        """Mark this item as recently accessed."""
        self.last_accessed = time.time()
        self.access_count += 1

    def to_message(self) -> dict:
        """Convert to the dict format the OpenAI API expects."""
        return {"role": self.role, "content": self.content}


# Quick test
item = ContextItem(content="Hello, I'm Alice.", role="user", source="user")
print(f"Content: {item.content!r}")
print(f"Tokens:  {item.token_count}")
print(f"ID:      {item.item_id}")
print(f"Pinned:  {item.pinned}")

### Step 2: The `SalienceScorer`

Imagine a news editor deciding which stories run on the front page. They weigh three factors: is the story about what readers care about *today*? (relevance). How fresh is it? (recency). How important is it regardless of timing? (importance). Our scorer does the same thing with numbers.

The salience score blends three signals:
1. **Embedding similarity**: How closely the item relates to the current query. We compute cosine similarity between their embeddings.
2. **Recency decay**: Items accessed recently score higher. We use exponential decay so scores drop quickly over time.
3. **Importance weight**: A static boost for high-priority items (like tool results or explicit user goals).

The formula: `score = w_rel * similarity + w_rec * recency + w_imp * importance`

In [ ]:
def get_embedding(text: str) -> list[float]:
    """Get an embedding vector from OpenAI."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text,
    )
    return response.data[0].embedding


def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    """Compute cosine similarity between two vectors."""
    a = np.array(vec_a)
    b = np.array(vec_b)
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    if norm == 0:
        return 0.0
    return float(dot / norm)


# Importance weights by source type
SOURCE_IMPORTANCE = {
    "system": 1.0,
    "tool": 0.8,
    "memory": 0.6,
    "user": 0.5,
    "assistant": 0.4,
}

Now we build the `SalienceScorer` class. It combines three signals into one number:
relevance (embedding similarity to the current query), recency (exponential decay since last access), and importance (a fixed weight based on item source). Pinned items get an infinite score so they never get evicted.

In [ ]:
class SalienceScorer:
    """Scores context items by relevance, recency, and importance."""

    def __init__(
        self,
        weight_relevance: float = 0.5,
        weight_recency: float = 0.3,
        weight_importance: float = 0.2,
        recency_half_life: float = 300.0,  # seconds until recency halves
    ):
        self.w_rel = weight_relevance
        self.w_rec = weight_recency
        self.w_imp = weight_importance
        self.half_life = recency_half_life

    def _recency_score(self, item: ContextItem) -> float:
        """Exponential decay based on time since last access."""
        age = time.time() - item.last_accessed
        # decay = 0.5^(age / half_life)
        return float(np.exp(-0.693 * age / self.half_life))

    def _importance_score(self, item: ContextItem) -> float:
        """Static importance based on item source."""
        return SOURCE_IMPORTANCE.get(item.source, 0.3)

    def score(
        self,
        item: ContextItem,
        query_embedding: list[float],
    ) -> float:
        """Compute the blended salience score for one item."""
        # Relevance: cosine similarity between item and current query
        if item.embedding is None:
            item.embedding = get_embedding(item.content)
        relevance = cosine_similarity(item.embedding, query_embedding)
        # Clamp to [0, 1] range
        relevance = max(0.0, relevance)

        recency = self._recency_score(item)
        importance = self._importance_score(item)

        return (
            self.w_rel * relevance
            + self.w_rec * recency
            + self.w_imp * importance
        )

    def score_all(
        self,
        items: list[ContextItem],
        query_embedding: list[float],
    ) -> list[ContextItem]:
        """Score all items and update their salience_score field."""
        for item in items:
            if not item.pinned:  # pinned items keep max score
                item.salience_score = self.score(item, query_embedding)
            else:
                item.salience_score = float("inf")
        return items

Let's test the scorer with a sample item and query. The score should be high because the item directly answers the query.

In [ ]:
# Quick test
scorer = SalienceScorer()
test_item = ContextItem(
    content="The capital of France is Paris.",
    role="assistant",
    source="assistant",
)
query_emb = get_embedding("What is the capital of France?")
score = scorer.score(test_item, query_emb)
print(f"Salience score: {score:.4f}")

### Step 3: The `EvictionEngine`

Picture a crowded bus. When a new passenger boards and there are no seats, someone has to stand or get off. A fair rule might be: the person who's been sitting longest gives up their seat (LRU). A smarter rule: the person whose destination is closest gets off first (importance-weighted). Our eviction engine uses both strategies.

The engine selects unpinned items for removal until enough tokens are freed for new content. It supports two policies:
- **LRU**: Remove items with the oldest `last_accessed` timestamp first.
- **Importance-weighted LRU**: Blend recency and salience score. Items that are both old *and* low-scoring go first.

In [ ]:
class EvictionEngine:
    """Selects items for eviction when the context window is full."""

    def __init__(self, policy: str = "importance_weighted_lru"):
        self.policy = policy

    def select_victims(
        self,
        items: list[ContextItem],
        tokens_to_free: int,
    ) -> list[ContextItem]:
        """Pick unpinned items for eviction to free at least `tokens_to_free` tokens."""
        candidates = [item for item in items if not item.pinned]

        if self.policy == "lru":
            # Sort by last_accessed ascending (oldest first)
            candidates.sort(key=lambda x: x.last_accessed)
        elif self.policy == "importance_weighted_lru":
            # Sort by salience_score ascending (lowest score = first to go)
            candidates.sort(key=lambda x: x.salience_score)
        else:
            raise ValueError(f"Unknown eviction policy: {self.policy}")

        victims = []
        freed = 0
        for candidate in candidates:
            if freed >= tokens_to_free:
                break
            victims.append(candidate)
            freed += candidate.token_count

        return victims


# Quick test
evictor = EvictionEngine(policy="importance_weighted_lru")
test_items = [
    ContextItem(content="Old low-score item", role="assistant", salience_score=0.1),
    ContextItem(content="Recent high-score item", role="user", salience_score=0.9),
    ContextItem(content="Pinned system prompt", role="system", pinned=True),
]
victims = evictor.select_victims(test_items, tokens_to_free=10)
print(f"Evicting {len(victims)} item(s): {[v.content for v in victims]}")

### Step 4: The External Memory Store

Think of a filing cabinet next to your desk. When your desk (the context window) gets too cluttered, you move papers to the cabinet. They're not gone. You can pull them back when you need them.

Our external store archives evicted items with their embeddings. When the scorer detects that an old item has become relevant again, the store retrieves it.

In [ ]:
class ExternalMemoryStore:
    """Archives evicted context items for later retrieval."""

    def __init__(self):
        self.archive: list[ContextItem] = []

    def store(self, item: ContextItem) -> None:
        """Archive an evicted item."""
        # Ensure the item has an embedding for later retrieval
        if item.embedding is None:
            item.embedding = get_embedding(item.content)
        self.archive.append(item)

    def retrieve(
        self,
        query_embedding: list[float],
        top_k: int = 3,
        min_similarity: float = 0.3,
    ) -> list[ContextItem]:
        """Find archived items most relevant to the current query."""
        if not self.archive:
            return []

        scored = []
        for item in self.archive:
            sim = cosine_similarity(item.embedding, query_embedding)
            if sim >= min_similarity:
                scored.append((sim, item))

        scored.sort(key=lambda x: x[0], reverse=True)
        return [item for _, item in scored[:top_k]]

    def remove(self, item_id: str) -> None:
        """Remove an item from the archive (e.g., after re-inserting it)."""
        self.archive = [i for i in self.archive if i.item_id != item_id]

    def __len__(self) -> int:
        return len(self.archive)


print("ExternalMemoryStore ready.")

### Step 5: The `ContextWindowManager`

This is the conductor of our orchestra. It coordinates the scorer, the evictor, and the store. On every turn it:
1. Scores all dynamic items against the current query.
2. Checks if the new item fits. If not, evicts the lowest-scoring items.
3. Retrieves relevant archived items when they match the current query.
4. Assembles the final prompt: pinned items first, then dynamic items sorted by score.

In [ ]:
class ContextWindowManager:
    """Orchestrates scoring, insertion, eviction, and prompt assembly."""

    def __init__(
        self,
        max_tokens: int = 4000,
        system_prompt: str = "You are a helpful assistant.",
        eviction_policy: str = "importance_weighted_lru",
        retrieval_top_k: int = 2,
    ):
        self.max_tokens = max_tokens
        self.scorer = SalienceScorer()
        self.evictor = EvictionEngine(policy=eviction_policy)
        self.external_store = ExternalMemoryStore()
        self.retrieval_top_k = retrieval_top_k

        # Pinned items stay in the window forever
        self.pinned_items: list[ContextItem] = []
        # Dynamic items compete for space
        self.dynamic_items: list[ContextItem] = []

        # Track eviction history for the demo
        self.eviction_log: list[dict] = []

        # Set up the system prompt as a pinned item
        sys_item = ContextItem(
            content=system_prompt,
            role="system",
            pinned=True,
            source="system",
        )
        self.pinned_items.append(sys_item)

    def _current_token_usage(self) -> int:
        """Total tokens currently in the window."""
        pinned = sum(item.token_count for item in self.pinned_items)
        dynamic = sum(item.token_count for item in self.dynamic_items)
        return pinned + dynamic

    def _available_tokens(self) -> int:
        """Tokens available for new content."""
        return self.max_tokens - self._current_token_usage()

    def _try_retrieve_from_archive(
        self, query_embedding: list[float]
    ) -> list[ContextItem]:
        """Check the external store for relevant evicted items."""
        retrieved = self.external_store.retrieve(
            query_embedding, top_k=self.retrieval_top_k
        )
        return retrieved

The `add_item` method is where the real work happens. It scores the new item, re-scores existing items against the new query, evicts low-scoring items if space is tight, and checks the archive for relevant evicted content to bring back.

In [ ]:
    def add_item(
        self,
        content: str,
        role: str,
        source: str = "user",
        query_text: str | None = None,
    ) -> None:
        """Add a new item to the context window. Evict if necessary."""
        item = ContextItem(content=content, role=role, source=source)
        query = query_text or content
        query_embedding = get_embedding(query)

        # Score the new item
        item.salience_score = self.scorer.score(item, query_embedding)
        item.embedding = get_embedding(item.content)

        # Re-score all existing dynamic items against the new query
        self.scorer.score_all(self.dynamic_items, query_embedding)

        # Check if we need to evict
        tokens_needed = item.token_count
        available = self._available_tokens()

        if tokens_needed > available:
            tokens_to_free = tokens_needed - available
            victims = self.evictor.select_victims(
                self.dynamic_items, tokens_to_free
            )
            for victim in victims:
                self.external_store.store(victim)
                self.dynamic_items.remove(victim)
                self.eviction_log.append({
                    "evicted": victim.content[:60],
                    "score": round(victim.salience_score, 4),
                    "tokens_freed": victim.token_count,
                })

        # Try to retrieve relevant archived items
        retrieved = self._try_retrieve_from_archive(query_embedding)
        for ret_item in retrieved:
            # Only re-insert if we have room
            if ret_item.token_count <= self._available_tokens() - tokens_needed:
                ret_item.touch()
                self.dynamic_items.append(ret_item)
                self.external_store.remove(ret_item.item_id)

        # Insert the new item
        self.dynamic_items.append(item)

Finally, `build_messages` assembles the prompt for the LLM. Pinned items go first, then dynamic items in chronological order. The `status` method gives a snapshot of token usage and item counts.

In [ ]:
    def build_messages(self) -> list[dict]:
        """Assemble the final message list for the LLM."""
        messages = []

        # Pinned items first (system prompt, etc.)
        for item in self.pinned_items:
            item.touch()
            messages.append(item.to_message())

        # Dynamic items sorted by creation time to preserve conversation order
        ordered = sorted(self.dynamic_items, key=lambda x: x.created_at)
        for item in ordered:
            item.touch()
            messages.append(item.to_message())

        return messages

    def status(self) -> dict:
        """Return a snapshot of the manager's state."""
        return {
            "max_tokens": self.max_tokens,
            "used_tokens": self._current_token_usage(),
            "available_tokens": self._available_tokens(),
            "pinned_items": len(self.pinned_items),
            "dynamic_items": len(self.dynamic_items),
            "archived_items": len(self.external_store),
            "total_evictions": len(self.eviction_log),
        }


print("ContextWindowManager ready.")

### Step 6: The Chat Function

Now we wire the manager to the LLM. The `chat` function adds the user message, builds the managed prompt, calls the API, and stores the response.

In [ ]:
def chat_with_managed_context(
    manager: ContextWindowManager,
    user_input: str,
    verbose: bool = False,
) -> str:
    """Send a message through the managed context window."""
    # Add user message to managed context
    manager.add_item(
        content=user_input,
        role="user",
        source="user",
    )

    # Build the prompt from managed context
    messages = manager.build_messages()

    if verbose:
        status = manager.status()
        print(f"  [Context: {status['used_tokens']}/{status['max_tokens']} tokens, "
              f"{status['dynamic_items']} dynamic, "
              f"{status['archived_items']} archived]")

    # Call the LLM
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        max_tokens=256,
    )

    assistant_text = response.choices[0].message.content

    # Add assistant response to managed context
    manager.add_item(
        content=assistant_text,
        role="assistant",
        source="assistant",
        query_text=user_input,  # score against the user's query
    )

    return assistant_text


print("Chat function ready.")

## Example Run

Let's watch the manager in action. We'll use a small token budget (800 tokens) to force evictions quickly. In a real application, you'd set this to match your model's context limit minus a reserve for the response.

We'll have a conversation about planning a trip. Early messages will get evicted as new ones arrive. Then we'll ask about something from an evicted message to see the retrieval loop in action.

In [ ]:
# Create a manager with a tight token budget to see eviction in action
manager = ContextWindowManager(
    max_tokens=800,
    system_prompt="You are a travel planning assistant. Keep replies under 3 sentences.",
    eviction_policy="importance_weighted_lru",
)

print(f"Starting state: {manager.status()}")
print("=" * 60)

conversation = [
    "Hi! I'm planning a trip to Japan in October.",
    "My budget is about $3,000 for two weeks.",
    "I love hiking and traditional temples.",
    "I'm also interested in the food scene in Osaka.",
    "What's the weather like in Kyoto in October?",
    "Can you recommend a good ryokan near Kyoto?",
    "I'm vegetarian, will that be a problem?",
    "What was my budget again?",  # tests retrieval of evicted info
]

for msg in conversation:
    print(f"\nUser: {msg}")
    reply = chat_with_managed_context(manager, msg, verbose=True)
    print(f"Agent: {reply}")

Let's inspect the final state. Check which items are in the active window, which were evicted, and the eviction log.

In [ ]:
print("Final status:")
for key, value in manager.status().items():
    print(f"  {key}: {value}")

print("\n--- Active context (dynamic items) ---")
for item in sorted(manager.dynamic_items, key=lambda x: x.created_at):
    label = item.role.upper()
    preview = item.content[:70] + ("..." if len(item.content) > 70 else "")
    print(f"  [{label}] score={item.salience_score:.3f} tokens={item.token_count}: {preview}")

print(f"\n--- Eviction log ({len(manager.eviction_log)} events) ---")
for entry in manager.eviction_log:
    print(f"  Evicted: {entry['evicted']!r} (score={entry['score']}, freed={entry['tokens_freed']} tokens)")

print(f"\n--- External store ({len(manager.external_store)} items) ---")
for item in manager.external_store.archive:
    preview = item.content[:70] + ("..." if len(item.content) > 70 else "")
    print(f"  [{item.role.upper()}] {preview}")

### Comparing Eviction Policies

Let's run the same conversation with pure LRU and importance-weighted LRU side by side. You'll see how the choice of policy affects what stays in the window.

In [ ]:
def run_conversation_with_policy(policy: str) -> dict:
    """Run the same conversation and return the final state."""
    mgr = ContextWindowManager(
        max_tokens=800,
        system_prompt="You are a travel planning assistant. Keep replies under 3 sentences.",
        eviction_policy=policy,
    )

    test_messages = [
        "Hi! I'm planning a trip to Japan in October.",
        "My budget is about $3,000 for two weeks.",
        "I love hiking and traditional temples.",
        "I'm also interested in the food scene in Osaka.",
        "What's the weather like in Kyoto in October?",
        "Can you recommend a good ryokan near Kyoto?",
    ]

    for msg in test_messages:
        chat_with_managed_context(mgr, msg)

    return {
        "policy": policy,
        "status": mgr.status(),
        "eviction_count": len(mgr.eviction_log),
        "active_items": [
            item.content[:50] for item in mgr.dynamic_items
        ],
        "archived_count": len(mgr.external_store),
    }


for policy in ["lru", "importance_weighted_lru"]:
    result = run_conversation_with_policy(policy)
    print(f"\nPolicy: {result['policy']}")
    print(f"  Evictions: {result['eviction_count']}")
    print(f"  Active items: {len(result['active_items'])}")
    print(f"  Archived: {result['archived_count']}")
    print(f"  Active content:")
    for content in result["active_items"]:
        print(f"    - {content}...")

## Tradeoffs

### When This Technique Wins

- **Long-running sessions**: Conversations with hundreds of turns stay coherent. The model always sees the most relevant context, not the most recent.
- **Predictable token budgets**: You control exactly how many tokens go to the LLM on each call. No surprises in cost or latency.
- **Critical instruction retention**: Pinned items guarantee that system prompts and key goals never get pushed out by chat volume.
- **Graceful degradation**: When capacity runs out, the system removes the least valuable items rather than cutting off mid-conversation.

### When It Breaks Down

- **Scoring overhead**: Every new message triggers embedding calls and re-scoring. For rapid-fire conversations, this adds latency. In our implementation, each turn needs 1-3 embedding API calls.
- **Imperfect salience estimates**: Embeddings measure semantic similarity, not logical importance. A message might be semantically distant from the current query but logically critical (e.g., "My flight leaves at 6 AM" when discussing dinner plans).
- **Lost conversational flow**: Evicted messages create gaps in the conversation history. The model might produce responses that feel disconnected because it can't see the full thread.
- **Complexity cost**: A naive buffer memory is 20 lines of code. This approach adds scoring, eviction, archival, and retrieval. That's more surface area for bugs.

## Further Reading

- Liu et al., ["Lost in the Middle: How Language Models Use Long Contexts,"](https://arxiv.org/abs/2307.03172) 2023. Shows that LLMs struggle with information placed in the middle of long contexts. This motivates explicit context management.
- Baddeley, ["The Episodic Buffer: A New Component of Working Memory?"](https://doi.org/10.1016/S1364-6613%2800%2901538-2) *Trends in Cognitive Sciences*, 2000. The foundational cognitive model of limited-capacity working memory with attentional control.
- Mu et al., ["Learning to Compress Prompts with Gist Tokens,"](https://arxiv.org/abs/2304.08467) 2023. Teaches models to compress verbose prompts into compact gist tokens. This enables more content in the same window.
- Modarressi et al., ["RET-LLM: Towards a General Read-Write Memory for LLMs,"](https://arxiv.org/abs/2305.14322) 2023. Proposes an explicit read/write memory layer for LLMs that complements context window management.
- [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings) for details on the embedding models used in this notebook.
- [tiktoken on GitHub](https://github.com/openai/tiktoken) for the token counting library.

---

*\u2190 Previous: [11 - Procedural Memory](../11_procedural_memory/) \u00b7 Next: [13 - Hierarchical Memory Layers](../13_hierarchical_memory_layers/) \u2192*

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: User-pinned items
Add a `pinned` boolean to `ContextItem`. Pinned items should never be evicted by the `EvictionEngine`. Modify `ContextWindowManager` to support a `pin(item_id)` method. Test by pinning a critical fact and running a long conversation that would normally evict it.

### Challenge 2: Eviction policy comparison
Run a 50-turn conversation twice: once with LRU eviction and once with importance-weighted eviction. After each run, ask 10 recall questions and score the answers. Compare which policy retains more useful information and report the recall difference.

### Challenge 3: External memory with vector search
Replace the basic `ExternalMemoryStore` with a ChromaDB-backed store that supports embedding-based retrieval. When a query arrives, check the external store for relevant evicted items and promote them back into the context window. This bridges to the tiered architecture in 13 Hierarchical Memory Layers.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--12-working-memory-context-window--working-memory-context-window)
